# 01 · Environment and models
**Objective:** confirm which real models can run here and measure a first call.

**Runs on:** a local Ollama server (GPU optional, slower on CPU) or the Hugging Face router with `HF_TOKEN`
(which may bill the token owner). All data is fictional.

In [1]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

from app.agents.llm import LLMError, ModelRegistry

registry = ModelRegistry()
status = {s.key: s for s in registry.status()}
CANDIDATES = ["qwen3-4b", "qwen2.5-7b", "mistral-7b", "hf-qwen3-8b"]
MODEL = next((k for k in CANDIDATES if status[k].available), None)
if MODEL is None:
    # There is deliberately no mock: results must come from a real model.
    raise RuntimeError("No model is reachable. Start Ollama and pull a model, or set HF_TOKEN.\n"
                       + "\n".join(f"{s.key}: {s.reason}" for s in status.values() if not s.available))
print(f"Using real model: {MODEL} ({registry.model(MODEL).model})")

Using real model: qwen3-4b (hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M)


## Environment check
Model availability comes from asking each provider, not from assumptions.

In [2]:
import shutil
for s in status.values():
    print(f"{s.key:14} {'available' if s.available else 'unavailable: ' + str(s.reason)}")
print()
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No NVIDIA GPU detected; expect CPU speeds.")

qwen3-4b       available
qwen2.5-7b     available
mistral-7b     available
nomic-embed    available
hf-qwen3-8b    unavailable: HF_TOKEN is not set

NVIDIA GeForce RTX 4060, 8188 MiB, 1358 MiB


## A real call, with validated output
The first call also loads the model, so it is slower than later ones.

In [3]:
from pydantic import BaseModel

class Fact(BaseModel):
    city: str
    employees: int | None

msg = [{"role": "user", "content": "Fictional note: Halden Labs has a team of 140 in Leeds. Extract the facts."}]
for attempt in ("cold or warm", "warm"):
    parsed, result = registry.structured(MODEL, msg, Fact, max_tokens=80)
    print(attempt, parsed, f"{result.latency_ms:.0f} ms", result.usage)
assert parsed.city.lower() == "leeds" and parsed.employees == 140, "Unexpected extraction"

cold or warm city='Leeds' employees=140 106384 ms Usage(input_tokens=34, output_tokens=23)


warm city='Leeds' employees=140 1186 ms Usage(input_tokens=34, output_tokens=23)


## Failures are explicit
A missing model raises a clear error. Nothing is silently substituted.

In [4]:
import tempfile, yaml
cfg = {"providers": {"ollama": {"base_url": registry.providers["ollama"].base_url}},
       "models": {"gone": {"provider": "ollama", "model": "no-such-model:latest", "capabilities": ["chat"]}}}
path = Path(tempfile.mkdtemp()) / "models.yaml"
path.write_text(yaml.safe_dump(cfg))
try:
    ModelRegistry(path).chat("gone", [{"role": "user", "content": "hi"}])
except LLMError as exc:
    print(f"{exc.category}: {exc}")

model_not_found: Model 'no-such-model:latest' is not available on 'ollama': {"error":{"message":"model 'no-such-model:latest' not found","type":"not_found_error","param":null,"code":null}}



**Expected:** a list of models with reasons for any that are unavailable, two validated extractions (the second
faster), and a `model_not_found` error.

**Limitations and next steps:** latency depends on hardware and on which model is already loaded. Next: notebook 02.